# AdaptiveCP — Main Entry Notebook

Spectral conformal prediction for black-box LLMs with adaptive sampling.

**Files:**
- `core.py` — config, scoring, sampling, calibration, prediction-set building
- `model.py` — LLM + sentence encoder loading + generation
- `data.py` — TriviaQA / WebQ / MMLU loaders
- `evaluate.py` — test-time evaluation + metrics + saving

**To run on Colab/Kaggle:** upload the whole `adaptive_cp/` folder, then open this notebook from inside it.

In [ ]:
!pip install -q sentence-transformers bitsandbytes datasets transformers accelerate

In [ ]:
import os, sys, random
import numpy as np
import torch

# Make local .py files importable when the notebook isn't in cwd (e.g. Colab)
HERE = os.path.dirname(os.path.abspath('__file__'))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
from core import SamplerConfig, MMLU_SUBJECTS
from model import load_model
from data import load_triviaqa, load_webquestions, load_mmlu
from evaluate import run_full_evaluation, print_comparison_table

## Load datasets and model

In [ ]:
DATASETS = {
    'triviaqa': load_triviaqa(SEED),
    'webq':     load_webquestions(SEED),
    'mmlu':     load_mmlu(SEED),
}
for name, samples in DATASETS.items():
    print(f'{name}: {len(samples)} samples')

In [ ]:
model, tokenizer = load_model()

## LofreeCP reference numbers (from the paper)
Used for side-by-side comparison only — not part of calibration.

In [ ]:
LOFREE_TRIVIAQA = {
    0.20: {'ECR': 80.1, 'SSC': 79.0, 'APSS': 2.19},
    0.25: {'ECR': 75.3, 'SSC': 74.5, 'APSS': 1.43},
    0.30: {'ECR': 70.3, 'SSC': 76.7, 'APSS': 1.08},
    0.35: {'ECR': 65.1, 'SSC': 78.5, 'APSS': 0.90},
    0.40: {'ECR': 60.0, 'SSC': 81.0, 'APSS': 0.75},
    0.45: {'ECR': 55.2, 'SSC': 84.1, 'APSS': 0.66},
}
LOFREE_WEBQ = {
    0.35: {'ECR': 65.1, 'SSC': 61.1, 'APSS': 5.33},
    0.40: {'ECR': 60.0, 'SSC': 60.0, 'APSS': 2.68},
    0.45: {'ECR': 55.1, 'SSC': 60.1, 'APSS': 1.60},
    0.50: {'ECR': 50.3, 'SSC': 59.9, 'APSS': 1.06},
}

ALPHAS_TRIVIAQA = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
ALPHAS_WEBQ     = [0.35, 0.40, 0.45, 0.50]

## Run full evaluation

In [ ]:
MODEL_NAME = 'qwen3b'
SAVE_DIR   = f'results/{MODEL_NAME}'

results_triviaqa = run_full_evaluation(
    model, tokenizer, DATASETS,
    dataset_name = 'triviaqa',
    alphas       = ALPHAS_TRIVIAQA,
    lofree_ref   = LOFREE_TRIVIAQA,
    cal_samples  = 500,
    test_samples = 500,
    save_dir     = SAVE_DIR,
    model_name   = MODEL_NAME,
)

results_webq = run_full_evaluation(
    model, tokenizer, DATASETS,
    dataset_name = 'webq',
    alphas       = ALPHAS_WEBQ,
    lofree_ref   = LOFREE_WEBQ,
    cal_samples  = 500,
    test_samples = 500,
    save_dir     = SAVE_DIR,
    model_name   = MODEL_NAME,
)

print_comparison_table(results_triviaqa, LOFREE_TRIVIAQA, 'TriviaQA')
print_comparison_table(results_webq,     LOFREE_WEBQ,     'WebQ')